# 少量提示模板的使用  
1.FewShotPromptTemplate ：需要结合PromptTemplate使用  
2.FewShotChatPromptTemplate：需要结合ChatPromptTemplate使用  
3.Example Selectors  





In [ ]:
from tempfile import template

import dotenv
import os
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI

#加载配置文件
dotenv.load_dotenv()
#读取配置文件
os.environ['OPENAI_BASE_URL'] = os.getenv('BASE_URL')
os.environ['OPENAI_API_KEY'] = os.getenv('API_KEY')

#创建对话模型
chatModel = ChatOpenAI(
    model="deepseek-chat",
    streaming=True
)

prompt = PromptTemplate(
    template='你好，请问{input}',
    input_variables = ['input']
)

promptValue = prompt.invoke({'input':'1+1=几'})

for chuck in chatModel.stream(promptValue):
    print(chuck.content,end='',flush=True)

In [ ]:
import dotenv
import os
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI

#加载配置文件
dotenv.load_dotenv()
#读取配置文件
os.environ['OPENAI_BASE_URL'] = os.getenv('BASE_URL')
os.environ['OPENAI_API_KEY'] = os.getenv('API_KEY')

#创建对话模型
chatModel = ChatOpenAI(
    model="deepseek-chat",
    streaming=True
)

prompt = PromptTemplate.from_template('input:{input}\noutput:{output}')
examples = [
    {"input": "北京天气怎么样", "output": "北京市"},
    {"input": "南京下雨吗", "output": "南京市"},
    {"input": "武汉热吗", "output": "武汉市"}
]
#FewShotPromptTemplate的实际作用过程
#1.根据普通的提示词模板PromptTemplate去解析example中的数据，并将解析出来的数据喂给ai，例如在当前项目当中就是去分别解析input和output的数据
#2.将上一步解析出来的模板数据作为新的input提交给我们的大模型，然后suffix中定义的值就作为下一个输入
#3.通过invoke方法完成模板的定义，这里的input会被suffix中定义的{input}给解析
#4.大模型输出结果
few_shot_prompt_template =  FewShotPromptTemplate(
    example_prompt=prompt,
    examples=examples,
    suffix='input:{input}\noutput:',
)

few_shot_prompt = few_shot_prompt_template.invoke({'input':'安徽怎么样'})

print(chatModel.invoke(few_shot_prompt.to_string()).content)

# 提示词模板的使用2 FewShotPromptTemplate

In [ ]:
import os
import dotenv
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_BASE_URL'] = os.getenv('BASE_URL')
os.environ['OPENAI_API_KEY'] = os.getenv('API_KEY')

prompt_template =  PromptTemplate.from_template('input:{input}\noutput:{output}')

examples = [
    {'input':'1[旺柴]5','output':'5'},
    {'input':'3[旺柴]5','output':'15'}
]
few_shot_template = FewShotPromptTemplate(
    examples=examples,
    suffix='input:{input},output:,直接说答案',
    example_prompt=prompt_template
)

few_shot_prompt_value = few_shot_template.invoke({'input':'2[旺柴]5'})

chatModel = ChatOpenAI(
    model="deepseek-chat",
)

print(chatModel.invoke(few_shot_prompt_value.to_string()).content)
